# NFL Fantasy Football Trade Analyzer

### Python | Pandas | NumPy | Scikit-learn | Plotly

This project develops a data-driven fantasy football trade analyzer that evaluates proposed trades using player production, usage, efficiency, positional value, and upcoming schedule difficulty.

The goal is to move beyond simple player rankings and create a framework for comparing the value of players on each side of a fantasy football trade.


## Project Objective

The objective of this project is to build a data-driven framework for evaluating fantasy football trades based on specific conditions of a league.

The analyzer will consider factors such as:
- Player fantasy production
- Target and snap share
- Touch and red-zone usage
- Player efficiency
- Recent performance trends
- Upcoming schedule difficulty
- Positional scarcity
- League scoring format

The final analysis will compare the value of players on each side of a proposed trade and identify the primary factors contributing to the difference.

In [ ]:
# Install NFLverse Pyton package
!pip install nflreadpy -q
# Data manipulation
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# NFL data
import nflreadpy as nfl



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.4 MB/s eta 0:00:00


## Data Sources

This project uses NFL data from NFLverse, an open-source collection of NFL datasets.

The primary data sources for this project include:

- **Weekly Player Statistics:** Passing, rushing, receiving, touchdowns, and other player-level statistics.
- **Player Information:** Player IDs, names, positions, teams, and other identifying information.
- **Roster and Depth Chart Data:** Player team assignments, depth charts, and player status.
- **Fantasy Football Data:** Fantasy scoring, rankings, and expected fantasy-point information.
- **Schedule Data:** Team schedules and opponents used to evaluate upcoming schedule difficulty.

### Primary Data Source

NFLverse: https://github.com/nflverse

The project uses `nflreadpy` to retrieve NFLverse data directly within Python.

In [ ]:
# Load Weekly Player Statistics
weekly_stats = nfl.load_player_stats(
    seasons=range(2022, 2027)
)

# Convert to a Pandas DataFrame
weekly_stats = weekly_stats.to_pandas()

# Display the first five rows
weekly_stats.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,...,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
0,00-0019596,T.Brady,Tom Brady,QB,QB,https://static.www.nfl.com/image/private/f_aut...,2022,1,REG,2022_01_TB_DAL,...,0,0,0,0,0,0,0,0,10.38,10.38
1,00-0022824,A.Lee,Andy Lee,P,SPEC,https://static.www.nfl.com/image/private/f_aut...,2022,1,REG,2022_01_KC_ARI,...,0,0,0,0,5,44,0,211,0.00,0.00
2,00-0023252,R.Gould,Robbie Gould,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2022,1,REG,2022_01_SF_CHI,...,0,0,0,0,0,0,0,0,0.00,0.00
3,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2022,1,REG,2022_01_GB_MIN,...,0,0,0,0,0,0,0,0,3.70,3.70
4,00-0023853,M.Prater,Matt Prater,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2022,1,REG,2022_01_KC_ARI,...,0,0,0,0,0,0,0,0,0.00,0.00


In [ ]:
# Display the size of the dataset
print("Dataset shape:", weekly_stats.shape)

# Display column names
print("\nColumn names:")
print(weekly_stats.columns.tolist())

# Display data types and non-null values
print("\nDataset information")
weekly_stats.info()

Dataset shape: (78173, 150)

Column names:
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'week', 'season_type', 'game_id', 'team', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions

## Initial Data Assessment

The weekly player statistics dataset contains 78,173 observations and 150 variables covering multiple NFL seasons.

The dataset includes:

- Player identification and position information
- Team and opponent information
- Passing statistics
- Rushing statistics
- Receiving statistics
- Player usage metrics
- Expected Points Added (EPA)
- Target share and air-yard share
- Fantasy points
- PPR fantasy points

The dataset contains both offensive and special-teams statistics. Since this project focuses on fantasy football trade evaluation, the analysis will primarily focus on offensive skill-position players: quarterbacks (QB), running backs (RB), wide receivers (WR), and tight ends (TE).

The large number of available variables allows us to select the metrics most relevant to player usage, production, efficiency, and fantasy value.

In [ ]:
# Check the positions represented in the dataset
weekly_stats["position"].value_counts()

,count
position,
LB,10707
WR,10614
CB,8336
RB,6640
DT,6457
DE,6419
TE,5281
SAF,5031
QB,2819


In [ ]:
# Define the fantasy-relevant positions
fantasy_positions = ["QB", "RB", "WR", "TE"]

#Keep only players at those positions
fantasy_stats = weekly_stats[
    weekly_stats["position"].isin(fantasy_positions)
].copy()

# Check the resulting dataset
print("Fantasy dataset shape:", fantasy_stats.shape)

fantasy_stats.head()

Fantasy dataset shape: (25354, 150)


,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,...,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
0,00-0019596,T.Brady,Tom Brady,QB,QB,https://static.www.nfl.com/image/private/f_aut...,2022,1,REG,2022_01_TB_DAL,...,0,0,0,0,0,0,0,0,10.38,10.38
3,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2022,1,REG,2022_01_GB_MIN,...,0,0,0,0,0,0,0,0,3.70,3.70
7,00-0026143,M.Ryan,Matt Ryan,QB,QB,https://static.www.nfl.com/image/private/f_aut...,2022,1,REG,2022_01_IND_HOU,...,0,0,0,0,0,0,0,0,15.28,15.28
8,00-0026158,J.Flacco,Joe Flacco,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2022,1,REG,2022_01_BAL_NYJ,...,0,0,0,0,0,0,0,0,14.36,14.36
10,00-0026197,C.Henne,Chad Henne,QB,QB,https://static.www.nfl.com/image/private/f_aut...,2022,1,REG,2022_01_KC_ARI,...,0,0,0,0,0,0,0,0,0.00,0.00


In [ ]:
#Select the columns we actually need
fantasy_cols = [
    "player_id",
    "player_name",
    "player_display_name",
    "position",
    "position_group",
    "season",
    "week",
    "season_type",
    "game_id",
    "team",
    "opponent_team",

    # Passing
    "passing_yards",
    "passing_tds",
    "passing_interceptions",
    "passing_epa",
    "passing_cpoe",

    # Rushing
    "carries",
    "rushing_yards",
    "rushing_tds",
    "rushing_epa",

    # Receiving
    "receptions",
    "targets",
    "receiving_yards",
    "receiving_tds",
    "receiving_epa",
    "target_share",
    "air_yards_share",

    # Fantasy production
    "fantasy_points",
    "fantasy_points_ppr"
]

fantasy_stats = fantasy_stats[fantasy_cols].copy()

print("Selected dataset shape:", fantasy_stats.shape)


fantasy_stats.head()


Selected dataset shape: (25354, 29)


,player_id,player_name,player_display_name,position,position_group,season,week,season_type,game_id,team,...,rushing_epa,receptions,targets,receiving_yards,receiving_tds,receiving_epa,target_share,air_yards_share,fantasy_points,fantasy_points_ppr
0,00-0019596,T.Brady,Tom Brady,QB,QB,2022,1,REG,2022_01_TB_DAL,TB,...,-1.437168,0,0,0,0,NaN,0.0,0.0,10.38,10.38
3,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,2022,1,REG,2022_01_GB_MIN,GB,...,0.000000,0,0,0,0,NaN,0.0,0.0,3.70,3.70
7,00-0026143,M.Ryan,Matt Ryan,QB,QB,2022,1,REG,2022_01_IND_HOU,IND,...,-4.417951,0,0,0,0,NaN,0.0,0.0,15.28,15.28
8,00-0026158,J.Flacco,Joe Flacco,QB,QB,2022,1,REG,2022_01_BAL_NYJ,NYJ,...,-1.230627,0,0,0,0,NaN,0.0,0.0,14.36,14.36
10,00-0026197,C.Henne,Chad Henne,QB,QB,2022,1,REG,2022_01_KC_ARI,KC,...,NaN,0,0,0,0,NaN,0.0,0.0,0.00,0.00


In [ ]:
# Check for missing values in each column
missing_values = fantasy_stats.isnull().sum()

# Display only columns that contain missing values
missing_values[missing_values > 0].sort_values(ascending=False)

,0
passing_cpoe,22558
passing_epa,22503
rushing_epa,15758
receiving_epa,6956


In [ ]:
#Check for missing values in important player identification columns
player_info_cols = [
    "player_id",
    "player_name",
    "player_display_name",
    "position",
    "team"
]

fantasy_stats[player_info_cols].isnull().sum()

,0
player_id,0
player_name,0
player_display_name,0
position,0
team,0


In [ ]:
# Statistical columns where missing weekly values can be treated as zero
zero_fill_cols = [
    "passing_cpoe",
    "passing_epa",
    "rushing_epa",
    "receiving_epa"
]

fantasy_stats[zero_fill_cols] = fantasy_stats[zero_fill_cols].fillna(0)

print("Missing values after cleaning:")

fantasy_stats[zero_fills_cols].isnull().sum()

Missing values after cleaning:


,0
passing_cpoe,0
passing_epa,0
rushing_epa,0
receiving_epa,0


In [ ]:
# Check for duplicate player-week records
duplicate_records = fantasy_stats.duplicated(
    subset=["player_id", "season", "week"],
    keep=False
)

print("Duplicate player-week records:", duplicate_records.sum())

# Display duplicates if any exist
fantasy_stats[duplicate_records].sort_values(
    ["player_id", "season", "week"]
).head(20)

Duplicate player-week records: 0


,player_id,player_name,player_display_name,position,position_group,season,week,season_type,game_id,team,...,rushing_epa,receptions,targets,receiving_yards,receiving_tds,receiving_epa,target_share,air_yards_share,fantasy_points,fantasy_points_ppr


In [ ]:
# Aggregate weekly statistics to the player-season level
season_stats = (
    fantasy_stats
    .groupby(
        [
        "player_id",
        "player_display_name",
        "position",
        "season"
    ],
    as_index=False
    )
    .agg(
        games=("week", "count"),

        passing_yards=("passing_yards", "sum"),
        passing_tds=("passing_tds", "sum"),
        passing_interceptions=("passing_interceptions", "sum"),
        passing_epa=("passing_epa", "sum"),
        passing_cpoe=("passing_cpoe", "mean"),

        carries= ("carries", "sum"),
        rushing_yards=("rushing_yards", "sum"),
        rushing_tds=("rushing_tds", "sum"),
        rushing_epa=("rushing_epa", "sum"),

        receptions=("receptions", "sum"),
        targets=("targets", "sum"),
        receiving_yards=("receiving_yards", "sum"),
        receiving_tds=("receiving_tds", "sum"),
        receiving_epa=("receiving_epa", "sum"),
        target_share=("target_share", "mean"),
        air_yards_share=("air_yards_share", "mean"),

        fantasy_points=("fantasy_points", "sum"),
        fantasy_points_ppr=("fantasy_points_ppr", "sum")
    )
)

print("Season-level dataset shape:", season_stats.shape)

season_stats.head()

Season-level dataset shape: (2806, 23)


,player_id,player_display_name,position,season,games,passing_yards,passing_tds,passing_interceptions,passing_epa,passing_cpoe,...,rushing_epa,receptions,targets,receiving_yards,receiving_tds,receiving_epa,target_share,air_yards_share,fantasy_points,fantasy_points_ppr
0,00-0019596,Tom Brady,QB,2022,18,5045,27,10,56.645483,0.218037,...,-20.469321,0,1,0,0,-4.726016,0.001916,0.003367,293.70,293.70
1,00-0022942,Philip Rivers,QB,2025,3,544,4,3,-11.195536,-3.753128,...,-1.146457,0,0,0,0,0.000000,0.000000,0.000000,31.66,31.66
2,00-0023459,Aaron Rodgers,QB,2022,17,3695,26,12,-5.150946,1.070413,...,-7.814974,0,0,0,0,0.000000,0.000000,0.000000,239.20,239.20
3,00-0023459,Aaron Rodgers,QB,2023,1,0,0,0,-2.031960,0.000000,...,0.000000,0,0,0,0,0.000000,0.000000,0.000000,0.00,0.00
4,00-0023459,Aaron Rodgers,QB,2024,17,3897,28,11,11.295307,-2.348851,...,4.280564,0,0,0,0,0.000000,0.000000,0.000000,256.58,256.58


In [ ]:
# Calculate fantasy production per game
season_stats["fantasy_points_per_game"] = (
    season_stats["fantasy_points"] / season_stats["games"]
)

season_stats["ppr_points_per_game"] = (
    season_stats["fantasy_points_ppr"]/ season_stats["games"]
)

season_stats[
    [
        "player_display_name",
        "position",
        "season",
        "games",
        "fantasy_points",
        "fantasy_points_per_game",
        "fantasy_points_ppr",
        "ppr_points_per_game"
    ]
].head(10)

,player_display_name,position,season,games,fantasy_points,fantasy_points_per_game,fantasy_points_ppr,ppr_points_per_game
0,Tom Brady,QB,2022,18,293.70,16.316667,293.70,16.316667
1,Philip Rivers,QB,2025,3,31.66,10.553333,31.66,10.553333
2,Aaron Rodgers,QB,2022,17,239.20,14.070588,239.20,14.070588
3,Aaron Rodgers,QB,2023,1,0.00,0.000000,0.00,0.000000
4,Aaron Rodgers,QB,2024,17,256.58,15.092941,256.58,15.092941
5,Aaron Rodgers,QB,2025,17,227.92,13.407059,228.92,13.465882
6,Aaron Rodgers,QB,2026,2,16.52,8.260000,16.52,8.260000
7,Marcedes Lewis,TE,2022,6,20.60,3.433333,26.60,4.433333
8,Marcedes Lewis,TE,2023,6,8.90,1.483333,12.90,2.150000
9,Marcedes Lewis,TE,2024,5,0.20,0.040000,1.20,0.240000


In [ ]:
# Calculate usage metrics per game
season_stats["targets_per_game"] = (
    season_stats["targets"] / season_stats["games"]
)

season_stats["receptions_per_game"]= (
    season_stats["receptions"] / season_stats["games"]
)

season_stats["carries_per_game"] = (
    season_stats["carries"]/ season_stats["games"]
)

season_stats[
    [
        "player_display_name",
        "position",
        "season",
        "games",
        "targets_per_game",
        "receptions_per_game",
        "carries_per_game",
        "ppr_points_per_game"

    ]
].head(10)

,player_display_name,position,season,games,targets_per_game,receptions_per_game,carries_per_game,ppr_points_per_game
0,Tom Brady,QB,2022,18,0.055556,0.000000,1.611111,16.316667
1,Philip Rivers,QB,2025,3,0.000000,0.000000,0.666667,10.553333
2,Aaron Rodgers,QB,2022,17,0.000000,0.000000,2.000000,14.070588
3,Aaron Rodgers,QB,2023,1,0.000000,0.000000,0.000000,0.000000
4,Aaron Rodgers,QB,2024,17,0.000000,0.000000,1.294118,15.092941
5,Aaron Rodgers,QB,2025,17,0.058824,0.058824,1.235294,13.465882
6,Aaron Rodgers,QB,2026,2,0.000000,0.000000,2.000000,8.260000
7,Marcedes Lewis,TE,2022,6,1.166667,1.000000,0.000000,4.433333
8,Marcedes Lewis,TE,2023,6,0.833333,0.666667,0.000000,2.150000
9,Marcedes Lewis,TE,2024,5,0.200000,0.200000,0.000000,0.240000


In [ ]:
# Calculate total opportunties
season_stats["opportunties"] = (
    season_stats["carries"] + season_stats["targets"]
)

# Calculate PPR points per opportunity
season_stats["ppr_points_per_opportunity"] = np.where(
    season_stats["opportunties"] > 0,
    season_stats["fantasy_points_ppr"] / season_stats["opportunties"],
    0
)

season_stats[
    [
        "player_display_name",
        "position",
        "season",
        "carries",
        "targets",
        "opportunties",
        "fantasy_points_ppr",
        "ppr_points_per_opportunity"
    ]
].head(10)

,player_display_name,position,season,carries,targets,opportunties,fantasy_points_ppr,ppr_points_per_opportunity
0,Tom Brady,QB,2022,29,1,30,293.70,9.790000
1,Philip Rivers,QB,2025,2,0,2,31.66,15.830000
2,Aaron Rodgers,QB,2022,34,0,34,239.20,7.035294
3,Aaron Rodgers,QB,2023,0,0,0,0.00,0.000000
4,Aaron Rodgers,QB,2024,22,0,22,256.58,11.662727
5,Aaron Rodgers,QB,2025,21,1,22,228.92,10.405455
6,Aaron Rodgers,QB,2026,4,0,4,16.52,4.130000
7,Marcedes Lewis,TE,2022,0,7,7,26.60,3.800000
8,Marcedes Lewis,TE,2023,0,5,5,12.90,2.580000
9,Marcedes Lewis,TE,2024,0,1,1,1.20,1.200000


In [ ]:
# Calculate weekly PPR scoring consistency
ppr_consistency = (
    fantasy_stats
    .groupby(
        [
            "player_id",
            "player_display_name",
            "position",
            "season"
        ]
    )["fantasy_points_ppr"]
    .std()
    .reset_index(name="ppr_points_std")
)

# Merge consistency back into the season-level dataset

season_stats = season_stats.merge(
    ppr_consistency,
    on=[
        "player_id",
        "player_display_name",
        "position",
        "season"
    ],
    how="left"
)

#Display the consistency metric

season_stats[
    [
        "player_display_name",
        "position",
        "season",
        "ppr_points_std"
    ]
].head(10)

,player_display_name,position,season,ppr_points_std
0,Tom Brady,QB,2022,7.054638
1,Philip Rivers,QB,2025,5.673459
2,Aaron Rodgers,QB,2022,3.855549
3,Aaron Rodgers,QB,2023,NaN
4,Aaron Rodgers,QB,2024,7.138301
5,Aaron Rodgers,QB,2025,6.740797
6,Aaron Rodgers,QB,2026,6.052834
7,Marcedes Lewis,TE,2022,3.958114
8,Marcedes Lewis,TE,2023,2.631919
9,Marcedes Lewis,TE,2024,0.536656


In [ ]:
# Calculate median weekly PPR scoring
ppr_median = (
    fantasy_stats
    .groupby(
        [
            "player_id",
            "player_display_name",
            "position",
            "season"
        ]
    )["fantasy_points_ppr"]
    .median()
    .reset_index(name="ppr_points_median")
)

# Merge the median scoring metric into season_stats

season_stats = season_stats.merge(
    ppr_median,
    on=[
        "player_id",
        "player_display_name",
        "position",
        "season"
    ],
    how="left"
)

season_stats[
    [
        "player_display_name",
        "position",
        "season",
        "ppr_points_per_game",
        "ppr_points_median",
        "ppr_points_std"
    ]
].head(10)

,player_display_name,position,season,ppr_points_per_game,ppr_points_median,ppr_points_std
0,Tom Brady,QB,2022,16.316667,15.61,7.054638
1,Philip Rivers,QB,2025,10.553333,7.78,5.673459
2,Aaron Rodgers,QB,2022,14.070588,15.22,3.855549
3,Aaron Rodgers,QB,2023,0.000000,0.00,NaN
4,Aaron Rodgers,QB,2024,15.092941,15.14,7.138301
5,Aaron Rodgers,QB,2025,13.465882,12.80,6.740797
6,Aaron Rodgers,QB,2026,8.260000,8.26,6.052834
7,Marcedes Lewis,TE,2022,4.433333,2.65,3.958114
8,Marcedes Lewis,TE,2023,2.150000,1.60,2.631919
9,Marcedes Lewis,TE,2024,0.240000,0.00,0.536656


In [25]:
# Sort weekly data chornologically
fantasy_stats = fantasy_stats.sort_values(
    ["player_id", "season", "week"]
)

recent_stats = (
    fantasy_stats
    .groupby(
        [
        "player_id",
        "player_display_name",
        "position",
        "season"
        ],
        group_keys=False
    )
    .tail(4)
)
# Calculate recent performance

recent_form = (
    recent_stats
    .groupby(
        [
            "player_id",
            "player_display_name",
            "position",
            "season"
        ],
        as_index=False
    )
    .agg(
        recent_games=("week", "count"),
        recent_ppr_per_game=("fantasy_points_ppr", "mean"),
        recent_targets_per_game=("targets", "mean"),
        recent_carries_per_game=("carries","mean")
    )
)

# Merge recent form into season_stats
season_stats = season_stats.merge(
    recent_form,
    on=[
        "player_id",
        "player_display_name",
        "position",
        "season"
    ],
    how="left"
)

# Display recent performance

season_stats[
    [
        "player_display_name",
        "position",
        "season",
        "recent_games",
        "recent_ppr_per_game",
        "recent_targets_per_game",
        "recent_carries_per_game"
    ]
].head(10)

,player_display_name,position,season,recent_games,recent_ppr_per_game,recent_targets_per_game,recent_carries_per_game
0,Tom Brady,QB,2022,4,19.605000,0.00,1.250000
1,Philip Rivers,QB,2025,3,10.553333,0.00,0.666667
2,Aaron Rodgers,QB,2022,4,12.985000,0.00,3.500000
3,Aaron Rodgers,QB,2023,1,0.000000,0.00,0.000000
4,Aaron Rodgers,QB,2024,4,17.085000,0.00,1.750000
5,Aaron Rodgers,QB,2025,4,10.615000,0.00,1.000000
6,Aaron Rodgers,QB,2026,2,8.260000,0.00,2.000000
7,Marcedes Lewis,TE,2022,4,4.125000,1.25,0.000000
8,Marcedes Lewis,TE,2023,4,2.125000,0.75,0.000000
9,Marcedes Lewis,TE,2024,4,0.300000,0.25,0.000000


In [27]:
# Calculate the change between recent and fuil-season PPR production
season_stats["recent_form_change"] = (
    season_stats["recent_ppr_per_game"]
    - season_stats["ppr_points_per_game"]
)

# Display the recent-form change
season_stats[
    [
        "player_display_name",
        "position",
        "season",
        "ppr_points_per_game",
        "recent_ppr_per_game",
        "recent_form_change"
    ]
].head(10)

,player_display_name,position,season,ppr_points_per_game,recent_ppr_per_game,recent_form_change
0,Tom Brady,QB,2022,16.316667,19.605000,3.288333
1,Philip Rivers,QB,2025,10.553333,10.553333,0.000000
2,Aaron Rodgers,QB,2022,14.070588,12.985000,-1.085588
3,Aaron Rodgers,QB,2023,0.000000,0.000000,0.000000
4,Aaron Rodgers,QB,2024,15.092941,17.085000,1.992059
5,Aaron Rodgers,QB,2025,13.465882,10.615000,-2.850882
6,Aaron Rodgers,QB,2026,8.260000,8.260000,0.000000
7,Marcedes Lewis,TE,2022,4.433333,4.125000,-0.308333
8,Marcedes Lewis,TE,2023,2.150000,2.125000,-0.025000
9,Marcedes Lewis,TE,2024,0.240000,0.300000,0.060000


In [29]:
# Find columns related to snap counts and red-zone usage
usage_columns = [
    col for col in weekly_stats.columns
    if any(term in col.lower() for term in ["snap","redzone", "red_zone"])
]

print("Usage-related columns:")
print(usage_columns)

Usage-related columns:
[]


In [30]:
# Check available NFLverse Loading functions
schedule_functions = [
    name for name in dir(nfl)
    if "schedule" in name.lower()
]

print("Schedule-related functions:")
print(schedule_functions)

Schedule-related functions:
['load_schedules']


In [34]:
# Load NFL schedules for the seasons in our dataset
schedules = nfl.load_schedules(
    seasons = range(2022, 2027)
)

# Convert to a Pandas DataFrame
schedules = schedules.to_pandas()

print("Schedule dataset shape:", schedules.shape)



print(schedules.columns.tolist())

Schedule dataset shape: (1411, 46)
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [35]:
# Calculate total PPR fantasy points scored against each defense
defense_fantasy = (
    fantasy_stats
    .groupby(
        ["season", "week", "opponent_team", "position"],
        as_index=False
    )
    .agg(
        opponent_ppr_points=("fantasy_points_ppr", "sum")
    )
)

# Display the defensive fantasy data
defense_fantasy.head(10)

,season,week,opponent_team,position,opponent_ppr_points
0,2022,1,ARI,QB,34.90
1,2022,1,ARI,RB,42.50
2,2022,1,ARI,TE,35.20
3,2022,1,ARI,WR,34.60
4,2022,1,ATL,QB,21.66
5,2022,1,ATL,RB,8.90
6,2022,1,ATL,TE,21.60
7,2022,1,ATL,WR,51.60
8,2022,1,BAL,QB,14.36
9,2022,1,BAL,RB,27.10


In [37]:
# Calcualte average PPR points allowed by each defense to each position
defense_sos = (
    defense_fantasy
    .groupby(
        ["season", "opponent_team", "position"],
        as_index=False
    )
    .agg(
        avg_ppr_allowed=("opponent_ppr_points", "mean")
    )
)

# Display the schedule difficulty data
defense_sos.head(12)

,season,opponent_team,position,avg_ppr_allowed
0,2022,ARI,QB,17.436471
1,2022,ARI,RB,27.917647
2,2022,ARI,TE,16.836471
3,2022,ARI,WR,30.594118
4,2022,ATL,QB,16.852941
5,2022,ATL,RB,21.164706
6,2022,ATL,TE,14.235294
7,2022,ATL,WR,34.511765
8,2022,BAL,QB,15.451111
9,2022,BAL,RB,20.000000


In [38]:
# Keep only 2026 regular-season games
current_schedule = schedules[
    (schedules["season"]== 2026) &
    (schedules["game_type"] == "REG")
].copy()

# Keep the columns needed for matchup analysis
current_schedule = current_schedule[
    [
        "game_id",
        "season",
        "week",
        "gameday",
        "away_team",
        "home_team"
    ]
].sort_values(["week", "gameday"])

print("2026 regular-season games:", len(current_schedule))

2026 regular-season games: 272


In [43]:
# Rebuild the team schedule cleanly
away = current_schedule[["game_id", "season", "week", "gameday", "away_team", "home_team"]].rename(
    columns={"away_team": "team", "home_team": "opponent"}
)

home = current_schedule[["game_id", "season", "week", "gameday", "home_team", "away_team"]].rename(
    columns={"home_team": "team", "away_team": "opponent"}
)

team_schedule = pd.concat([away, home], ignore_index=True)
team_schedule = team_schedule.sort_values(["team", "week"])

print("Team-game records:", len(team_schedule))
team_schedule.head()

Team-game records: 544


,game_id,season,week,gameday,team,opponent
10,2026_01_ARI_LAC,2026,1,2026-09-13,ARI,LAC
299,2026_02_SEA_ARI,2026,2,2026-09-20,ARI,SEA
42,2026_03_ARI_SF,2026,3,2026-09-27,ARI,SF
55,2026_04_ARI_NYG,2026,4,2026-10-04,ARI,NYG
346,2026_05_DET_ARI,2026,5,2026-10-11,ARI,DET


In [46]:
player_schedule = fantasy_stats[
    fantasy_stats["season"] == 2026
][
    ["player_id", "player_display_name", "position", "team", "week"]
].drop_duplicates()

print(player_schedule.columns.tolist())

['player_id', 'player_display_name', 'position', 'team', 'week']


In [47]:
player_schedule = player_schedule.merge(
    team_schedule,
    on=["team", "week"],
    how="left"
)

player_schedule.head(10)

,player_id,player_display_name,position,team,week,game_id,season,gameday,opponent
0,00-0023459,Aaron Rodgers,QB,PIT,1,2026_01_ATL_PIT,2026,2026-09-13,ATL
1,00-0023459,Aaron Rodgers,QB,PIT,2,2026_02_PIT_NE,2026,2026-09-20,NE
2,00-0026498,Matthew Stafford,QB,LA,1,2026_01_SF_LA,2026,2026-09-10,SF
3,00-0026498,Matthew Stafford,QB,LA,2,2026_02_NYG_LA,2026,2026-09-21,NYG
4,00-0029604,Kirk Cousins,QB,LV,1,2026_01_MIA_LV,2026,2026-09-13,MIA
5,00-0029604,Kirk Cousins,QB,LV,2,2026_02_LV_LAC,2026,2026-09-20,LAC
6,00-0030279,Keenan Allen,WR,IND,1,2026_01_BAL_IND,2026,2026-09-13,BAL
7,00-0030279,Keenan Allen,WR,IND,2,2026_02_IND_KC,2026,2026-09-20,KC
8,00-0030506,Travis Kelce,TE,KC,1,2026_01_DEN_KC,2026,2026-09-14,DEN
9,00-0030506,Travis Kelce,TE,KC,2,2026_02_IND_KC,2026,2026-09-20,IND


In [48]:
player_schedule = player_schedule.merge(
    defense_sos,
    left_on=["season", "opponent", "position"],
    right_on=["season", "opponent_team", "position"],
    how="left"
)

player_schedule.head(10)

,player_id,player_display_name,position,team,week,game_id,season,gameday,opponent,opponent_team,avg_ppr_allowed
0,00-0023459,Aaron Rodgers,QB,PIT,1,2026_01_ATL_PIT,2026,2026-09-13,ATL,ATL,18.233333
1,00-0023459,Aaron Rodgers,QB,PIT,2,2026_02_PIT_NE,2026,2026-09-20,NE,NE,8.790000
2,00-0026498,Matthew Stafford,QB,LA,1,2026_01_SF_LA,2026,2026-09-10,SF,SF,9.000000
3,00-0026498,Matthew Stafford,QB,LA,2,2026_02_NYG_LA,2026,2026-09-21,NYG,NYG,20.690000
4,00-0029604,Kirk Cousins,QB,LV,1,2026_01_MIA_LV,2026,2026-09-13,MIA,MIA,21.040000
5,00-0029604,Kirk Cousins,QB,LV,2,2026_02_LV_LAC,2026,2026-09-20,LAC,LAC,18.350000
6,00-0030279,Keenan Allen,WR,IND,1,2026_01_BAL_IND,2026,2026-09-13,BAL,BAL,31.250000
7,00-0030279,Keenan Allen,WR,IND,2,2026_02_IND_KC,2026,2026-09-20,KC,KC,20.650000
8,00-0030506,Travis Kelce,TE,KC,1,2026_01_DEN_KC,2026,2026-09-14,DEN,DEN,9.450000
9,00-0030506,Travis Kelce,TE,KC,2,2026_02_IND_KC,2026,2026-09-20,IND,IND,18.850000


In [49]:
player_schedule["matchup_score"] = (
  player_schedule
  .groupby("position")["avg_ppr_allowed"]
  .transform(lambda x: x.rank(pct=True) * 100 )
)

player_schedule[
    [
    "player_display_name",
    "position",
    "week",
    "opponent",
    "avg_ppr_allowed",
    "matchup_score"
    ]
].head(10)



,player_display_name,position,week,opponent,avg_ppr_allowed,matchup_score
0,Aaron Rodgers,QB,1,ATL,18.233333,58.333333
1,Aaron Rodgers,QB,2,NE,8.790000,10.897436
2,Matthew Stafford,QB,1,SF,9.000000,15.384615
3,Matthew Stafford,QB,2,NYG,20.690000,78.846154
4,Kirk Cousins,QB,1,MIA,21.040000,82.051282
5,Kirk Cousins,QB,2,LAC,18.350000,66.025641
6,Keenan Allen,WR,1,BAL,31.250000,53.833866
7,Keenan Allen,WR,2,KC,20.650000,8.306709
8,Travis Kelce,TE,1,DEN,9.450000,25.316456
9,Travis Kelce,TE,2,IND,18.850000,89.873418


In [52]:
# Upcoming 4-week schedule
current_week = 3
upcoming_schedule = player_schedule[
    player_schedule["week"].between(current_week + 1, current_week + 4)
].copy()

upcoming_sos = (
    upcoming_schedule
    .groupby(
        ["player_id", "player_display_name", "position"],
        as_index=False
    )
    .agg(
        upcoming_games=("week", "count"),
        avg_matchup_score=("matchup_score", "mean"),
        avg_ppr_allowed=("avg_ppr_allowed","mean")
    )
)

upcoming_sos.head(10)

,player_id,player_display_name,position,upcoming_games,avg_matchup_score,avg_ppr_allowed


In [54]:
current_players = (
    fantasy_stats[fantasy_stats["season"] == 2026]
    .sort_values("week")
    .groupby("player_id")
    .tail(1)
    [["player_id", "player_display_name", "position", "team"]]
)

future_schedule = current_players.merge(
    team_schedule[team_schedule["week"].between(4,7)],
    on="team",
    how="left"
)

future_schedule.head(10)

,player_id,player_display_name,position,team,game_id,season,week,gameday,opponent
0,00-0041511,Omar Cooper Jr.,WR,NYJ,2026_04_NYJ_CHI,2026,4,2026-10-04,CHI
1,00-0041511,Omar Cooper Jr.,WR,NYJ,2026_05_CLE_NYJ,2026,5,2026-10-11,CLE
2,00-0041511,Omar Cooper Jr.,WR,NYJ,2026_06_NYJ_NE,2026,6,2026-10-18,NE
3,00-0041511,Omar Cooper Jr.,WR,NYJ,2026_07_MIA_NYJ,2026,7,2026-10-25,MIA
4,00-0041503,Nate Boerkircher,TE,JAX,2026_04_JAX_CIN,2026,4,2026-10-04,CIN
5,00-0041503,Nate Boerkircher,TE,JAX,2026_05_PHI_JAX,2026,5,2026-10-11,PHI
6,00-0041503,Nate Boerkircher,TE,JAX,2026_06_HOU_JAX,2026,6,2026-10-18,HOU
7,00-0038489,Ben VanSumeren,RB,KC,2026_04_KC_LV,2026,4,2026-10-04,LV
8,00-0038489,Ben VanSumeren,RB,KC,2026_06_LAC_KC,2026,6,2026-10-18,LAC
9,00-0038489,Ben VanSumeren,RB,KC,2026_07_KC_SEA,2026,7,2026-10-25,SEA


In [57]:
future_schedule = future_schedule.merge(
    defense_sos,
    left_on=["season", "opponent", "position"],
    right_on=["season", "opponent_team", "position"],
    how="left"
)

future_schedule.head(10)

,player_id,player_display_name,position,team,game_id,season,week,gameday,opponent,opponent_team,avg_ppr_allowed
0,00-0041511,Omar Cooper Jr.,WR,NYJ,2026_04_NYJ_CHI,2026,4,2026-10-04,CHI,CHI,28.45
1,00-0041511,Omar Cooper Jr.,WR,NYJ,2026_05_CLE_NYJ,2026,5,2026-10-11,CLE,CLE,39.10
2,00-0041511,Omar Cooper Jr.,WR,NYJ,2026_06_NYJ_NE,2026,6,2026-10-18,NE,NE,28.15
3,00-0041511,Omar Cooper Jr.,WR,NYJ,2026_07_MIA_NYJ,2026,7,2026-10-25,MIA,MIA,23.30
4,00-0041503,Nate Boerkircher,TE,JAX,2026_04_JAX_CIN,2026,4,2026-10-04,CIN,CIN,18.50
5,00-0041503,Nate Boerkircher,TE,JAX,2026_05_PHI_JAX,2026,5,2026-10-11,PHI,PHI,5.10
6,00-0041503,Nate Boerkircher,TE,JAX,2026_06_HOU_JAX,2026,6,2026-10-18,HOU,HOU,11.10
7,00-0038489,Ben VanSumeren,RB,KC,2026_04_KC_LV,2026,4,2026-10-04,LV,LV,16.45
8,00-0038489,Ben VanSumeren,RB,KC,2026_06_LAC_KC,2026,6,2026-10-18,LAC,LAC,16.50
9,00-0038489,Ben VanSumeren,RB,KC,2026_07_KC_SEA,2026,7,2026-10-25,SEA,SEA,16.15


In [58]:
future_schedule["matchup_score"] = (
    future_schedule
    .groupby("position")["avg_ppr_allowed"]
    .transform(lambda x: x.rank(pct=True) * 100 )
)

future_schedule[
    [
        "player_display_name",
        "position",
        "week",
        "opponent",
        "avg_ppr_allowed",
        "matchup_score"
    ]
].head(10)

,player_display_name,position,week,opponent,avg_ppr_allowed,matchup_score
0,Omar Cooper Jr.,WR,4,CHI,28.45,36.413902
1,Omar Cooper Jr.,WR,5,CLE,39.10,74.486572
2,Omar Cooper Jr.,WR,6,NE,28.15,32.938389
3,Omar Cooper Jr.,WR,7,MIA,23.30,17.298578
4,Nate Boerkircher,TE,4,CIN,18.50,88.017751
5,Nate Boerkircher,TE,5,PHI,5.10,4.289941
6,Nate Boerkircher,TE,6,HOU,11.10,47.189349
7,Ben VanSumeren,RB,4,LV,16.45,31.658291
8,Ben VanSumeren,RB,6,LAC,16.50,35.050251
9,Ben VanSumeren,RB,7,SEA,16.15,21.984925


In [60]:
upcoming_sos = (
    future_schedule
    .groupby(
        ["player_id", "player_display_name", "position"],
        as_index=False
    )
    .agg(
        upcoming_games=("week", "count"),
        avg_matchup_score=("matchup_score", "mean"),
        avg_ppr_allowed=("avg_ppr_allowed", "mean")
    )
)

upcoming_sos.head(10)

,player_id,player_display_name,position,upcoming_games,avg_matchup_score,avg_ppr_allowed
0,00-0023459,Aaron Rodgers,QB,4,63.811728,19.630000
1,00-0026498,Matthew Stafford,QB,4,56.481481,17.895000
2,00-0029604,Kirk Cousins,QB,4,29.398148,13.137500
3,00-0030279,Keenan Allen,WR,4,60.150079,33.737500
4,00-0030506,Travis Kelce,TE,3,61.193294,13.900000
5,00-0030565,Geno Smith,QB,4,58.950617,17.225000
6,00-0031381,Davante Adams,WR,4,57.365719,33.350000
7,00-0031408,Mike Evans,WR,4,40.560821,28.329167
8,00-0031503,Jameis Winston,QB,4,70.061728,21.527500
9,00-0031588,Stefon Diggs,WR,3,58.267509,34.300000


In [62]:
# Combine player value + schedule
player_value = season_stats.merge(
    upcoming_sos,
    on=["player_id", "player_display_name", "position"],
    how="left"
)

player_value[
    [
        "player_display_name",
        "position",
        "season",
        "ppr_points_per_game",
        "recent_ppr_per_game",
        "recent_form_change",
        "avg_matchup_score"
    ]
].head(10)

,player_display_name,position,season,ppr_points_per_game,recent_ppr_per_game,recent_form_change,avg_matchup_score
0,Tom Brady,QB,2022,16.316667,19.605000,3.288333,NaN
1,Philip Rivers,QB,2025,10.553333,10.553333,0.000000,NaN
2,Aaron Rodgers,QB,2022,14.070588,12.985000,-1.085588,63.811728
3,Aaron Rodgers,QB,2023,0.000000,0.000000,0.000000,63.811728
4,Aaron Rodgers,QB,2024,15.092941,17.085000,1.992059,63.811728
5,Aaron Rodgers,QB,2025,13.465882,10.615000,-2.850882,63.811728
6,Aaron Rodgers,QB,2026,8.260000,8.260000,0.000000,63.811728
7,Marcedes Lewis,TE,2022,4.433333,4.125000,-0.308333,NaN
8,Marcedes Lewis,TE,2023,2.150000,2.125000,-0.025000,NaN
9,Marcedes Lewis,TE,2024,0.240000,0.300000,0.060000,NaN


In [63]:
# Keep only the current 2026 player values
current_values = player_value[
    player_value["season"] == 2026
].copy()

# Use a neutral matchup score when schedule is unavailable
current_values["avg_matchup_score"] = (
    current_values["avg_matchup_score"].fillna(50)
)

# Display the number of current player records
print("Current player records:", len(current_values))

# Preview the curent player values
current_values.head(10)

Current player records: 415


,player_id,player_display_name,position,season,games,passing_yards,passing_tds,passing_interceptions,passing_epa,passing_cpoe,...,ppr_points_median,recent_games,recent_ppr_per_game,recent_targets_per_game,recent_carries_per_game,recent_change,recent_form_change,upcoming_games,avg_matchup_score,avg_ppr_allowed
6,00-0023459,Aaron Rodgers,QB,2026,2,408,1,1,-29.247575,-2.588042,...,8.26,2,8.26,0.0,2.0,0.0,0.0,4.0,63.811728,19.630000
27,00-0026498,Matthew Stafford,QB,2026,2,482,4,2,13.022811,1.292059,...,15.54,2,15.54,0.0,3.0,0.0,0.0,4.0,56.481481,17.895000
68,00-0029604,Kirk Cousins,QB,2026,2,413,6,3,6.365936,1.799739,...,17.01,2,17.01,0.0,4.5,0.0,0.0,4.0,29.398148,13.137500
88,00-0030279,Keenan Allen,WR,2026,2,0,0,0,0.000000,0.000000,...,5.35,2,5.35,5.5,0.0,0.0,0.0,4.0,60.150079,33.737500
98,00-0030506,Travis Kelce,TE,2026,2,0,0,0,0.000000,0.000000,...,17.60,2,17.60,8.0,0.0,0.0,0.0,3.0,61.193294,13.900000
110,00-0030565,Geno Smith,QB,2026,2,462,1,0,14.039089,9.350420,...,12.44,2,12.44,0.0,5.5,0.0,0.0,4.0,58.950617,17.225000
147,00-0031381,Davante Adams,WR,2026,2,0,0,0,0.000000,0.000000,...,22.55,2,22.55,8.0,0.0,0.0,0.0,4.0,57.365719,33.350000
154,00-0031408,Mike Evans,WR,2026,2,0,0,0,0.000000,0.000000,...,12.65,2,12.65,5.0,0.0,0.0,0.0,4.0,40.560821,28.329167
166,00-0031503,Jameis Winston,QB,2026,1,111,0,1,-16.490465,-25.332596,...,2.54,1,2.54,0.0,2.0,0.0,0.0,4.0,70.061728,21.527500
184,00-0031588,Stefon Diggs,WR,2026,2,0,0,0,0.000000,0.000000,...,18.60,2,18.60,7.5,0.0,0.0,0.0,3.0,58.267509,34.300000


In [67]:
# Create a function to evaluate both sides of a fantasy trade
def evaluate_trade(give_players, receive_players):
  # Find players being given

  give = current_values[
      current_values["player_display_name"].isin(give_players)
  ].copy()

  # Find players being received
  receive = current_values[
      current_values["player_display_name"].isin(receive_players)
  ].copy()

  # Calculate total trade value for each side
  give_value = give["trade_value_score"].sum()
  receive_value = receive["trade_value_score"].sum()

  # Calculate the difference between the two sides
  value_difference = receive_value - give_value

  # Create a summary table
  trade_summary = pd.DataFrame({
      "Trade Side": ["Giving", "Receiving"],
      "Total Trade Value": [give_value, receive_value]
  })

  print(f"Players giving: {', '.join(give_players)}")
  print(f"Players receiving: {','.join(receive_players)}")
  print(f"\nValue difference: {value_difference:.2f}")

  return trade_summary

In [68]:
# Calculate position-relative production scores
current_values["production_score"] = (
    current_values
    .groupby("position")["ppr_points_per_game"]
    .rank(pct=True) * 100
)

# Calculate position-relative recent performance scores
current_values["recent_score"] = (
    current_values
    .groupby("position")["recent_ppr_per_game"]
    .rank(pct=True) * 100
)

# Calculate consistency scores
# Lower standard deviation means more consistent scoring.
current_values["consistency_score"] = (
    current_values
    .groupby("position")["ppr_points_std"]
    .rank(pct=True, ascending=False) * 100
)

# Replace missing component scores with a neutral value
current_values["recent_score"] = current_values["recent_score"].fillna(50)
current_values["consistency_score"] = current_values["consistency_score"].fillna(50)
current_values["avg_matchup_score"] = current_values["avg_matchup_score"].fillna(50)

# Combine the components into the overall trade value score
current_values["trade_value_score"] = (
    current_values["production_score"] * 0.45 +
    current_values["recent_score"] * 0.25 +
    current_values["consistency_score"] * 0.15 +
    current_values["avg_matchup_score"] * 0.15
)

# Display the highest-valued players
current_values[
    [
        "player_display_name",
        "position",
        "ppr_points_per_game",
        "recent_ppr_per_game",
        "avg_matchup_score",
        "trade_value_score"
    ]
].sort_values(
    "trade_value_score",
    ascending=False
).head(15)

,player_display_name,position,ppr_points_per_game,recent_ppr_per_game,avg_matchup_score,trade_value_score
2360,Bryce Young,QB,27.76,27.76,72.633745,86.576880
2323,Zay Flowers,WR,26.00,26.00,63.941548,86.272519
751,Mike Gesicki,TE,18.80,18.80,53.599606,84.018202
1554,Amon-Ra St. Brown,WR,31.95,31.95,61.269089,83.856195
1364,Nico Collins,WR,21.20,21.20,61.196682,83.404649
781,Josh Allen,QB,38.24,38.24,15.946502,82.846521
2150,Parker Washington,WR,18.05,18.05,55.371248,82.825320
2751,Tyler Shough,QB,22.79,22.79,43.672840,82.687290
1893,Trey McBride,TE,21.30,21.30,53.328402,82.686760
2349,Jahmyr Gibbs,RB,28.45,28.45,68.299832,82.261300


In [77]:
# Test the trade analyzer with an example trade
trade_result = evaluate_trade(
    give_players=["Christian McCaffrey", "Rashod Bateman"],
    receive_players=["Jonathon Taylor", "Deebo Samuel"]
)

trade_result

Players giving: Christian McCaffrey, Rashod Bateman
Players receiving: Jonathon Taylor,Deebo Samuel

Value difference: -140.52


,Trade Side,Total Trade Value
0,Giving,140.515106
1,Receiving,0.000000


## 12. Data Visualizations

The visual analysis focuses on three key dimensions of fantasy trade value:

- Overall PPR production
- Recent player performance
- Upcoming matchup difficulty

Interactive visualizations are used to identify differences between players and examine which factors contribute to overall trade value.

In [82]:
# Select the top 20 current players by trade value
top_players = current_values. sort_values("trade_value_score", ascending=False).head(20)


# Horizontal bar chart

fig =px.bar(
    top_players.sort_values("trade_value_score"),
    x="trade_value_score",
    y="player_display_name",
    color="position",
    orientation="h",
    hover_data = [
        "position",
        "ppr_points_per_game",
        "recent_ppr_per_game",
        "avg_matchup_score"
    ],
    title="Top 20 Current Players by Fantasy Trade Value",
    labels={
        "trade_value_score": "Trade Value Score",
        "player_display_name": "Player",
        "position": "Position"
    }
)

# Chart readability
fig.update_layout(
    height=700,
    xaxis_title="Trade Value Score",
    yaxis_title="Player",
    legend_title="Position"
)

# Display the visualization
fig.show()

In [83]:
fig = px.scatter(
    current_values,
    x="ppr_points_per_game",
    y="recent_ppr_per_game",
    color="position",
    hover_name="player_display_name",
    hover_data=[
        "season",
        "ppr_points_per_game",
        "recent_ppr_per_game",
        "recent_form_change",
        "trade_value_score"
    ],
    title="Season Production vs. Recent Performance",
    labels={
        "ppr_points_per_game": "Season PPR Points per Game",
        "recent_ppr_per_game": "Recent PPR Points per Game",
        "position": "Position",
        "trade_value_score": "Trade Value Score"
    }
)

# Reference line
max_value = max(
    current_values["ppr_points_per_game"].max(),
    current_values["recent_ppr_per_game"].max()
)

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=max_value,
    y1=max_value,
    line=dict(dash="dash")
)

#Chart layout
fig.update_layout(
    height=650
)

# Display the visualization
fig.show()

In [89]:
# Trade Test Visualization

# Select example players for each side of the trade
give_players = [
    "Mike Evans",
    "Derrick Henry"
]

receive_players = [
    "Davante Adams",
    "Travis Kelce"
]

# Pull the player values for each side
give_data = current_values[
    current_values["player_display_name"].isin(give_players)
].copy()

receive_data = current_values[
    current_values["player_display_name"].isin(receive_players)
].copy()

# Label each player by trade side
give_data["trade_side"] = "Giving"
receive_data["trade_side"] = "Receiving"

# Combine both sides into one dataset
trade_players = pd.concat(
    [give_data, receive_data],
    ignore_index=True
)

# Create an interactive trade comparison chart
fig = px.bar(
    trade_players,
    x="player_display_name",
    y="trade_value_score",
    color="trade_side",
    hover_data=[
        "position",
        "ppr_points_per_game",
        "recent_ppr_per_game",
        "avg_matchup_score",
        "trade_value_score"
    ],
    title="Trade Test Visualization",
    labels={
        "player_display_name": "Player",
        "trade_value_score": "Trade Value Score",
        "trade_side": "Trade Side"
    }
)

# Improve chart readability
fig.update_layout(
    height=600,
    xaxis_title="Player",
    yaxis_title="Trade Value Score",
    legend_title="Trade Side"
)

# Display the interactive visualization
fig.show()



## 13. Results & Key Findings

The fantasy football trade analyzer combines player production, recent performance, scoring consistency, and upcoming matchup difficulty into a single trade value score.

The analysis demonstrates that:

- PPR production is the largest component of the trade value score.
- Recent performance helps identify players whose current production differs from their overall season performance.
- Scoring consistency provides additional context when comparing players with similar production.
- Upcoming matchup difficulty can increase or decrease a player's overall trade value.
- The trade evaluation function can compare multiple players on each side of a proposed fantasy football trade.

The trade testing framework can also compare multiple players on each side of a proposed trade rather than evaluating players individually. This provides a structured way to compare the combined calculated value of the players being given and received.

## 14. Limitations

This analysis has several limitations:

- The trade value weights are analyst-defined and have not been statistically optimized against actual fantasy trade outcomes.
- The current-season sample contains only the weeks available in the dataset, so early-season player values may change as more games are played.
- Historical defensive PPR allowed is used to estimate matchup difficulty, but defensive performance can change throughout a season.
- Injuries, depth-chart changes, coaching decisions, and changes in player roles are not directly modeled.
- League-specific roster settings, waiver-wire availability, keeper rules, and individual team needs are not included in the current score.
- Players with very few games can have unstable production and consistency measurements.
- The current version focuses on production, recent form, consistency, and schedule difficulty rather than incorporating every possible fantasy football variable.

## 15. Conclusion

This project developed a data-driven fantasy football trade analyzer using NFL player statistics and schedule data.

The framework transforms weekly player data into season-level metrics, evaluates recent performance and scoring consistency, estimates upcoming matchup difficulty, and combines these factors into a transparent trade value score.

The final trade evaluation function allows users to enter the players they are giving and receiving and compare the total calculated value of both sides.

The project demonstrates how Python, Pandas, statistical analysis, and interactive visualization can be combined to create a practical analytics tool for fantasy football decision-making.

Future improvements could include league-specific scoring settings, roster needs, injury information, player projections, and historical validation of the trade value model.